# MNIST Data Exploration Notebook

This notebook explores the MNIST dataset to understand:
- Image characteristics
- Label distribution
- Data quality
- Preprocessing needs

In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import sys
import os

# Add project root to path so we can import our modules
sys.path.insert(0, os.path.abspath('..'))

# Import our data loader
from src.mnist_classifier.load_data import load_mnist, get_dataset_info

# Configure visualization settings
plt.style.use('default')  # Use default style for consistency
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['font.size'] = 12

print("Imports successful! 🎉")

In [ ]:
# Load the MNIST dataset
print("Loading MNIST dataset...")
(x_train, y_train), (x_test, y_test) = load_mnist()

# Get dataset information
info = get_dataset_info(x_train, y_train, x_test, y_test)

print(f"\n✅ Dataset loaded successfully!")
print(f"Training samples: {info['train_samples']:,}")
print(f"Test samples: {info['test_samples']:,}")
print(f"Image shape: {info['image_shape']}")
print(f"Memory usage: {info['memory_usage_mb']['total']:.2f} MB")

In [ ]:
def plot_digit_samples(images, labels, n_samples=25, title="Sample Digits"):
    """
    Plot a grid of sample digit images with their labels.
    """
    # Calculate grid dimensions
    n_cols = int(np.sqrt(n_samples))
    n_rows = int(np.ceil(n_samples / n_cols))

    # Create figure
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 10))
    fig.suptitle(title, fontsize=16)

    # Flatten axes array for easy iteration
    axes = axes.ravel()

    # Plot each sample
    for i in range(n_samples):
        # Select random image
        idx = np.random.randint(0, len(images))

        # Plot image
        axes[i].imshow(images[idx], cmap='gray')
        axes[i].set_title(f'Label: {labels[idx]}')
        axes[i].axis('off')

    # Hide any unused subplots
    for i in range(n_samples, len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

# Display random samples
plot_digit_samples(x_train, y_train, n_samples=25,
                  title="Random Training Samples")

In [ ]:
def analyze_pixel_distribution(images, sample_size=1000):
    """
    Analyze the distribution of pixel values in the dataset.
    """
    # Sample images for efficiency
    sample_indices = np.random.choice(len(images), sample_size, replace=False)
    sample_images = images[sample_indices]

    # Flatten all pixels
    all_pixels = sample_images.flatten()

    # Create figure with subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Histogram of pixel values
    ax1.hist(all_pixels, bins=50, density=True, alpha=0.7, color='blue', edgecolor='black')
    ax1.set_xlabel('Pixel Value')
    ax1.set_ylabel('Density')
    ax1.set_title('Distribution of Pixel Values')
    ax1.axvline(all_pixels.mean(), color='red', linestyle='--',
                label=f'Mean: {all_pixels.mean():.1f}')
    ax1.axvline(all_pixels.std(), color='green', linestyle='--',
                label=f'Std: {all_pixels.std():.1f}')
    ax1.legend()

    # Box plot by digit class
    pixel_by_class = []
    for digit in range(10):
        digit_images = images[sample_indices[y_train[sample_indices] == digit]]
        if len(digit_images) > 0:
            pixel_by_class.append(digit_images.flatten())

    ax2.boxplot(pixel_by_class, labels=range(10))
    ax2.set_xlabel('Digit Class')
    ax2.set_ylabel('Pixel Value')
    ax2.set_title('Pixel Value Distribution by Digit Class')

    plt.tight_layout()
    plt.show()

    # Print statistics
    print("📊 Pixel Value Statistics:")
    print(f"Min value: {all_pixels.min()}")
    print(f"Max value: {all_pixels.max()}")
    print(f"Mean value: {all_pixels.mean():.2f}")
    print(f"Std deviation: {all_pixels.std():.2f}")
    print(f"Percentage of zero pixels (black): {(all_pixels == 0).mean() * 100:.1f}%")
    print(f"Percentage of max pixels (white): {(all_pixels == 255).mean() * 100:.1f}%")

analyze_pixel_distribution(x_train)

In [ ]:
def plot_class_distribution(y_train, y_test):
    """
    Visualize the distribution of classes in train and test sets.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # Training set distribution
    train_counts = np.bincount(y_train)
    ax1.bar(range(10), train_counts, color='skyblue', edgecolor='black')
    ax1.set_xlabel('Digit')
    ax1.set_ylabel('Count')
    ax1.set_title('Training Set Class Distribution')
    ax1.set_xticks(range(10))

    # Add count labels on bars
    for i, count in enumerate(train_counts):
        ax1.text(i, count + 100, str(count), ha='center')

    # Test set distribution
    test_counts = np.bincount(y_test)
    ax2.bar(range(10), test_counts, color='lightcoral', edgecolor='black')
    ax2.set_xlabel('Digit')
    ax2.set_ylabel('Count')
    ax2.set_title('Test Set Class Distribution')
    ax2.set_xticks(range(10))

    # Add count labels on bars
    for i, count in enumerate(test_counts):
        ax2.text(i, count + 20, str(count), ha='center')

    plt.tight_layout()
    plt.show()

    # Calculate and display balance metrics
    train_balance = train_counts.std() / train_counts.mean()
    test_balance = test_counts.std() / test_counts.mean()

    print("⚖️ Class Balance Analysis:")
    print(f"Training set - Coefficient of variation: {train_balance:.3f}")
    print(f"Test set - Coefficient of variation: {test_balance:.3f}")
    print(f"(Lower values indicate better balance, 0 = perfect balance)")

plot_class_distribution(y_train, y_test)

In [ ]:
def examine_digit_variations(images, labels, digit=7, n_samples=10):
    """
    Show variations in how a specific digit is written.
    """
    # Find all instances of the digit
    digit_indices = np.where(labels == digit)[0]

    # Sample some instances
    sample_indices = np.random.choice(digit_indices,
                                    min(n_samples, len(digit_indices)),
                                    replace=False)

    # Plot the variations
    fig, axes = plt.subplots(2, 5, figsize=(12, 6))
    fig.suptitle(f'Different Ways People Write "{digit}"', fontsize=16)

    axes = axes.ravel()
    for i, idx in enumerate(sample_indices):
        axes[i].imshow(images[idx], cmap='gray')
        axes[i].axis('off')

        # Calculate some basic statistics for each image
        img = images[idx]
        center_of_mass_y = np.sum(np.arange(28) * img.sum(axis=1)) / img.sum()
        center_of_mass_x = np.sum(np.arange(28) * img.sum(axis=0)) / img.sum()

        axes[i].plot(center_of_mass_x, center_of_mass_y, 'r+', markersize=10)

    plt.tight_layout()
    plt.show()

    print(f"Note: Red crosses show the center of mass of each digit.")
    print(f"This helps visualize how digits are positioned within the 28x28 frame.")

# Examine variations for each digit
for digit in [1, 7, 9]:  # These digits often have interesting variations
    examine_digit_variations(x_train, y_train, digit=digit)

In [ ]:
def analyze_image_quality(images, labels, n_samples=1000):
    """
    Analyze image quality metrics like contrast and centering.
    """
    # Sample for efficiency
    sample_indices = np.random.choice(len(images), n_samples, replace=False)
    sample_images = images[sample_indices]
    sample_labels = labels[sample_indices]

    # Calculate metrics for each image
    contrasts = []
    center_offsets = []
    ink_percentages = []

    for img in sample_images:
        # Contrast (std of pixel values)
        contrasts.append(img.std())

        # Center offset
        y_coords, x_coords = np.mgrid[0:28, 0:28]
        center_y = np.sum(y_coords * img) / np.sum(img) if np.sum(img) > 0 else 14
        center_x = np.sum(x_coords * img) / np.sum(img) if np.sum(img) > 0 else 14
        offset = np.sqrt((center_x - 14)**2 + (center_y - 14)**2)
        center_offsets.append(offset)

        # Ink percentage (non-zero pixels)
        ink_percentages.append((img > 0).mean() * 100)

    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # Contrast distribution
    axes[0, 0].hist(contrasts, bins=30, alpha=0.7, color='green', edgecolor='black')
    axes[0, 0].set_xlabel('Contrast (Std Dev)')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].set_title('Image Contrast Distribution')

    # Center offset distribution
    axes[0, 1].hist(center_offsets, bins=30, alpha=0.7, color='orange', edgecolor='black')
    axes[0, 1].set_xlabel('Distance from Center (pixels)')
    axes[0, 1].set_ylabel('Count')
    axes[0, 1].set_title('Digit Centering Distribution')

    # Ink percentage distribution
    axes[1, 0].hist(ink_percentages, bins=30, alpha=0.7, color='purple', edgecolor='black')
    axes[1, 0].set_xlabel('Ink Coverage (%)')
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].set_title('Ink Coverage Distribution')

    # Examples of outliers
    # Find low contrast image
    low_contrast_idx = sample_indices[np.argmin(contrasts)]
    axes[1, 1].imshow(images[low_contrast_idx], cmap='gray')
    axes[1, 1].set_title(f'Low Contrast Example (Label: {labels[low_contrast_idx]})')
    axes[1, 1].axis('off')

    plt.tight_layout()
    plt.show()

    print("📊 Image Quality Statistics:")
    print(f"Average contrast: {np.mean(contrasts):.2f}")
    print(f"Average center offset: {np.mean(center_offsets):.2f} pixels")
    print(f"Average ink coverage: {np.mean(ink_percentages):.1f}%")

analyze_image_quality(x_train, y_train)

In [ ]:
def preview_augmentations(image):
    """
    Preview potential data augmentations for training.
    """
    from scipy.ndimage import rotate, shift

    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    fig.suptitle('Potential Data Augmentations', fontsize=16)

    # Original
    axes[0, 0].imshow(image, cmap='gray')
    axes[0, 0].set_title('Original')
    axes[0, 0].axis('off')

    # Slight rotation
    rotated = rotate(image, angle=15, reshape=False)
    axes[0, 1].imshow(rotated, cmap='gray')
    axes[0, 1].set_title('Rotated +15°')
    axes[0, 1].axis('off')

    # Slight shift
    shifted = shift(image, shift=(2, -2))
    axes[0, 2].imshow(shifted, cmap='gray')
    axes[0, 2].set_title('Shifted')
    axes[0, 2].axis('off')

    # Add noise
    noisy = image + np.random.normal(0, 10, image.shape)
    noisy = np.clip(noisy, 0, 255)
    axes[0, 3].imshow(noisy, cmap='gray')
    axes[0, 3].set_title('With Noise')
    axes[0, 3].axis('off')

    # Brightness adjustment
    bright = np.clip(image * 1.2, 0, 255)
    axes[1, 0].imshow(bright, cmap='gray')
    axes[1, 0].set_title('Brighter')
    axes[1, 0].axis('off')

    # Contrast adjustment
    mean = image.mean()
    contrast = np.clip((image - mean) * 1.5 + mean, 0, 255)
    axes[1, 1].imshow(contrast, cmap='gray')
    axes[1, 1].set_title('Higher Contrast')
    axes[1, 1].axis('off')

    # Slight blur (simulate different pen styles)
    from scipy.ndimage import gaussian_filter
    blurred = gaussian_filter(image, sigma=0.5)
    axes[1, 2].imshow(blurred, cmap='gray')
    axes[1, 2].set_title('Slight Blur')
    axes[1, 2].axis('off')

    # Elastic deformation preview
    axes[1, 3].text(0.5, 0.5, 'Elastic\nDeformation\n(Advanced)',
                    ha='center', va='center', transform=axes[1, 3].transAxes)
    axes[1, 3].axis('off')

    plt.tight_layout()
    plt.show()

# Show augmentation possibilities
sample_idx = np.random.randint(0, len(x_train))
print(f"Showing augmentation preview for a '{y_train[sample_idx]}':")
preview_augmentations(x_train[sample_idx])

In [ ]:
def create_exploration_summary():
    """
    Summarize key findings from our exploration.
    """
    print("📋 MNIST Dataset Exploration Summary")
    print("=" * 50)

    print("\n1. Dataset Characteristics:")
    print(f"   - Total samples: {len(x_train) + len(x_test):,}")
    print(f"   - Image dimensions: 28x28 pixels (784 features)")
    print(f"   - Grayscale values: 0 (black) to 255 (white)")
    print(f"   - Highly imbalanced: No (all classes ~10% ± 1%)")

    print("\n2. Key Observations:")
    print("   - Most pixels are black (background)")
    print("   - Digits are generally well-centered")
    print("   - Significant variation in writing styles")
    print("   - Good contrast between digits and background")

    print("\n3. Preprocessing Recommendations:")
    print("   ✓ Normalize pixel values to [0, 1] range")
    print("   ✓ Reshape for CNN: (28, 28) → (28, 28, 1)")
    print("   ✓ One-hot encode labels for classification")
    print("   ✓ Consider data augmentation for better generalization")

    print("\n4. Potential Challenges:")
    print("   - Similar looking digits (e.g., 1 vs 7, 3 vs 8)")
    print("   - Varying stroke thickness and styles")
    print("   - Some digits are off-center or rotated")

    print("\n5. Next Steps:")
    print("   → Implement preprocessing pipeline")
    print("   → Create train/validation split")
    print("   → Design CNN architecture")
    print("   → Train and evaluate model")

create_exploration_summary()